# Setup

In [6]:
# workspace_namespace = "broad-firecloud-dsde-methods"
# workspace_name = "kj-rdtest-test"
# submission_ids = ["e0160046-238e-479e-ab74-35d8f4d799e3"]
# billing_table = "broad-dsde-methods.Methods_billing_dump.gcp_billing_export_v1_009C7D_923007_219A6F"

In [7]:
workspace_namespace = "LR_GNOMAD_1_CO-AoU_TALK" # @param {type:"string"}
workspace_name = "LR_GNOMAD-AoU_TALK_CNV-Benchmarking" # @param {type:"string"}
submission_ids = ["432c323f-fbb1-4e35-9716-e78c65df2d62"] # @param {type:"raw"}
billing_table = "project.dataset.gcp_billing_export_v1_XXXXXX" # @param {type:"string"}

`billing_table` is the BigQuery billing export table for this workspace's billing account, in `project.dataset.table` form. It must be obtained from whoever administers the Terra billing project (Terra billing project Owner + GCP Billing Account Owner/Admin) - see [How to set up spend reporting in Terra](https://support.terra.bio/hc/en-us/articles/10026441196187-How-to-set-up-spend-reporting-in-Terra-GCP-Terra-Billing-project-owners) and [How to retrieve detailed workflow cost information on GCP](https://support.terra.bio/hc/en-us/articles/360036932671-How-to-retrieve-detailed-workflow-cost-information-on-GCP).

In [8]:
import subprocess
import json
import requests
import pandas as pd

In [9]:
oauth_token = subprocess.check_output("gcloud auth print-access-token", shell=True, text=True).strip()

headers = {
    'Authorization': f'Bearer {oauth_token}',
    'Content-Type': 'application/json',
}

# Get workflow IDs from submissions

In [10]:
workflows = []
for sub_id in submission_ids:
    endpoint = f"https://api.firecloud.org/api/workspaces/{workspace_namespace}/{workspace_name}/submissions/{sub_id}"
    submission_details = requests.get(endpoint, headers=headers).json()

    for workflow in submission_details["workflows"]:
        entity = workflow.get("workflowEntity", {})
        workflows.append({
            "submissionId": sub_id,
            "methodConfigurationName": submission_details["methodConfigurationName"],
            "workflowId": workflow["workflowId"],
            "entityType": entity.get("entityType"),
            "entityName": entity.get("entityName"),
        })

# Query BigQuery for cost

In [11]:
workflow_labels = [f'"cromwell-{w["workflowId"]}"' for w in workflows]

query = f"""
SELECT l.value AS workflow_label, SUM(cost) AS cost, ANY_VALUE(currency) AS currency
FROM `{billing_table}`, UNNEST(labels) AS l
WHERE l.key = "cromwell-workflow-id" AND l.value IN ({', '.join(workflow_labels)})
GROUP BY workflow_label
"""

result = subprocess.check_output(["bq", "query", "--use_legacy_sql=false", "--format=json", query], text=True)
cost_by_label = {row["workflow_label"]: row for row in json.loads(result)}

for w in workflows:
    row = cost_by_label.get(f'cromwell-{w["workflowId"]}', {})
    w["cost"] = float(row.get("cost", 0) or 0)
    w["currency"] = row.get("currency")

pd.DataFrame(workflows)

CalledProcessError: Command '['bq', 'query', '--use_legacy_sql=false', '--format=json', '\nSELECT l.value AS workflow_label, SUM(cost) AS cost, ANY_VALUE(currency) AS currency\nFROM `project.dataset.gcp_billing_export_v1_XXXXXX`, UNNEST(labels) AS l\nWHERE l.key = "cromwell-workflow-id" AND l.value IN ("cromwell-1268e605-060a-47d4-aa16-d10f04ec40d7", "cromwell-e2589c78-cf2d-479d-994e-718835d533a1", "cromwell-4f095a49-4866-4a72-92fa-e22501ed594e", "cromwell-5c78b99e-4faf-4813-98b7-37025fc408f0", "cromwell-1e1e373c-e23d-4674-8f79-790aee78d622")\nGROUP BY workflow_label\n']' returned non-zero exit status 1.